In [36]:

#SCRAP LIST OF BILLIONAIRES FROM FORBES.COM/BILLIONAIRES

In [33]:
import os
import re
import pandas as pd

COLS = ["Rank", "Name", "NetWorth_USD_B", "Age", "Country", "Source", "Industry"]
WORTH = re.compile(r"^\$\s*([\d.,]+)\s*([BM])$")


def read_text(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8-sig", "utf-16"):
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    return raw.decode("latin-1")


def tokenize(text):
    text = (text.replace("\u25b2", "").replace("\u25bc", "")
                .replace("\xa0", " "))
    tokens = []
    for part in re.split(r"[\t\r\n]+", text):
        for piece in re.split(r"(\$\s*[\d.,]+\s*[BM])", part):
            piece = piece.strip()
            if not piece:
                continue
            m = re.match(r"^#?(\d+)\.?\s+(\D.*)$", piece)
            if m:
                tokens += [m.group(1), m.group(2)]
            else:
                tokens.append(piece)
    return tokens


def parse(tokens):
    records = []
    for i, t in enumerate(tokens):
        m = WORTH.match(t)
        if not m:
            continue

        # rank is 2 tokens back, or 3 when the name has an "& family" part
        rank_idx = None
        for k in (2, 3):
            if i - k >= 0 and re.fullmatch(r"#?(\d[\d,]*)\.?", tokens[i - k]):
                rank_idx = i - k
                break
        if rank_idx is None:
            continue

        rank = int(re.sub(r"\D", "", tokens[rank_idx]))
        name = " ".join(tokens[rank_idx + 1:i])

        def g(k):
            return tokens[i + k] if i + k < len(tokens) else None

        worth = float(m.group(1).replace(",", ""))
        if m.group(2) == "M":
            worth /= 1000

        age = g(1)
        if age and re.fullmatch(r"\d{1,3}", age):
            age_val, off = int(age), 2
        elif age in ("-", "\u2014", "N/A", "n/a"):
            age_val, off = None, 2
        else:
            age_val, off = None, 1

        records.append([rank, name, worth, age_val,
                        g(off), g(off + 1), g(off + 2)])
    return records


def main(path):
    text = read_text(path)
    tokens = tokenize(text)
    records = parse(tokens)
    print(f"Read {len(tokens)} pieces of text from the file")

    if not records:
        print("No rows recognised. First 15 pieces:")
        for t in tokens[:15]:
            print(repr(t))
        raise ValueError("Send those lines to Claude and the parser can be adjusted.")

    df = pd.DataFrame(records, columns=COLS)
    df = df.sort_values(["Rank", "Name"]).reset_index(drop=True)
    df.to_csv("forbes_billionaires.csv", index=False)
    print(f"Saved {len(df)} rows to {os.path.abspath('forbes_billionaires.csv')}")
    return df


df = main(r"C:\Users\sakat\paste.txt")
df

Read 37295 pieces of text from the file
Saved 3428 rows to C:\Users\sakat\forbes_billionaires.csv


,Rank,Name,NetWorth_USD_B,Age,Country,Source,Industry
0,1,Elon Musk,839.0,54.0,United States,"Tesla, SpaceX",Technology
1,2,Larry Page,257.0,52.0,United States,Google,Technology
2,3,Sergey Brin,237.0,52.0,United States,Google,Technology
3,4,Jeff Bezos,224.0,62.0,United States,Amazon,Technology
4,5,Mark Zuckerberg,222.0,41.0,United States,Facebook,Technology
...,...,...,...,...,...,...,...
3423,3332,Tulay Kazanci,1.0,70.0,Turkey,Diversified,Diversified
3424,3332,Vladimir Ivanov,1.0,55.0,Russia,Yandex,Technology
3425,3332,Vladimir Melnikov,1.0,78.0,Russia,Fast fashion,Fashion & Retail
3426,3332,William Franke,1.0,88.0,United States,Low-cost airlines,Diversified


In [25]:
import time
import hashlib
import re
import tkinter as tk


def get_clip():
    root = tk.Tk()
    root.withdraw()
    try:
        return root.clipboard_get()
    except tk.TclError:
        return ""
    finally:
        root.destroy()


def collect(path=r"C:\Users\sakat\paste.txt", pages=68, idle_minutes=10):
    seen = set()
    added = 0
    last_new = time.time()
    print(f"Ready. On Forbes: click a page, Ctrl+A, Ctrl+C. Waiting for {pages} pages...")
    try:
        while added < pages and time.time() - last_new < idle_minutes * 60:
            text = get_clip()
            rows = len(re.findall(r"\$\s*[\d.,]+\s*[BM]", text))
            key = hashlib.md5(text.encode("utf-8")).hexdigest()
            if rows >= 20 and key not in seen:
                seen.add(key)
                with open(path, "a", encoding="utf-8") as f:
                    f.write("\n" + text + "\n")
                added += 1
                print(f"Page {added} added ({rows} rows)")
                last_new = time.time()
            time.sleep(1)
    except KeyboardInterrupt:
        print("Stopped.")
    print(f"Done: {added} pages added.")


collect()

Ready. On Forbes: click a page, Ctrl+A, Ctrl+C. Waiting for 68 pages...
Page 1 added (60 rows)
Page 2 added (60 rows)
Page 3 added (60 rows)
Page 4 added (60 rows)
Page 5 added (60 rows)
Page 6 added (60 rows)
Page 7 added (60 rows)
Page 8 added (60 rows)
Page 9 added (60 rows)
Page 10 added (60 rows)
Page 11 added (60 rows)
Page 12 added (60 rows)
Page 13 added (60 rows)
Page 14 added (60 rows)
Page 15 added (60 rows)
Page 16 added (60 rows)
Page 17 added (60 rows)
Page 18 added (60 rows)
Page 19 added (60 rows)
Page 20 added (60 rows)
Page 21 added (60 rows)
Page 22 added (60 rows)
Page 23 added (60 rows)
Page 24 added (60 rows)
Page 25 added (60 rows)
Page 26 added (60 rows)
Page 27 added (60 rows)
Page 28 added (60 rows)
Page 29 added (60 rows)
Page 30 added (60 rows)
Page 31 added (60 rows)
Page 32 added (60 rows)
Page 33 added (60 rows)
Page 34 added (60 rows)
Page 35 added (60 rows)
Page 36 added (60 rows)
Page 37 added (60 rows)
Page 38 added (60 rows)
Page 39 added (60 rows)
P

In [31]:
raw = pd.DataFrame(parse(tokenize(read_text(r"C:\Users\sakat\paste.txt"))), columns=COLS)
print("parsed:", len(raw), "| after removing duplicates:", len(df))

have = sorted(set(df["Rank"]))
print("Big gaps in ranks (start-end):")
for a, b in zip(have, have[1:]):
    if b - a > 10:
        print(a + 1, "-", b - 1)
    

parsed: 3428 | after removing duplicates: 3160
Big gaps in ranks (start-end):
445 - 454
462 - 476
488 - 497
499 - 508
510 - 519
529 - 541
543 - 556
568 - 587
596 - 611
613 - 622
624 - 644
654 - 663
665 - 676
695 - 711
731 - 748
750 - 761
763 - 779
781 - 805
807 - 822
824 - 836
838 - 851
853 - 868
870 - 890
892 - 907
909 - 933
935 - 953
955 - 971
973 - 983
985 - 1010
1012 - 1043
1045 - 1073
1075 - 1107
1109 - 1136
1138 - 1162
1164 - 1188
1190 - 1222
1224 - 1250
1252 - 1284
1286 - 1324
1326 - 1375
1377 - 1405
1407 - 1439
1441 - 1503
1505 - 1559
1561 - 1610
1612 - 1675
1677 - 1754
1756 - 1833
1835 - 1912
1914 - 1981
1983 - 2051
2053 - 2176
2178 - 2273
2275 - 2385
2387 - 2480
2482 - 2599
2601 - 2711
2713 - 2857
2859 - 3016
3018 - 3184
3186 - 3331
